# Initialization
Turning the symlinks into proper paths the system can access. Finding the 16mixmux16 firmware.

In [1]:
!hostname -I
!ip -4 addr show

192.168.2.100 
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
4: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qlen 1000
    inet 192.168.2.100/24 brd 192.168.2.255 scope global eth0:1
       valid_lft forever preferred_lft forever


In [2]:
import os, shutil
repo = "/home/xilinx/jupyter_notebooks/qick"
src = os.path.join(repo, "qick_lib", "qick")     # standard repo layout
print("exists:", os.path.exists(src))

!grep -rl "mixmux16" {src}                        # should list files, e.g. asm_v2.py

import qick
print(qick.__file__)                              # confirm the kernel uses this same copy

shutil.make_archive("/home/xilinx/qick_pkg", "zip", os.path.join(repo, "qick_lib"), "qick")

exists: True
/home/xilinx/jupyter_notebooks/qick/qick_lib/qick/asm_v2.py
/home/xilinx/jupyter_notebooks/qick/qick_lib/qick/drivers/generator.py
/home/xilinx/jupyter_notebooks/qick/qick_lib/qick/drivers/__pycache__/generator.cpython-310.pyc
/home/xilinx/jupyter_notebooks/qick/qick_lib/qick/drivers/__pycache__/generator.cpython-38.pyc
/home/xilinx/jupyter_notebooks/qick/qick_lib/qick/__pycache__/asm_v2.cpython-38.pyc
/home/xilinx/jupyter_notebooks/qick/qick_lib/qick/__pycache__/asm_v2.cpython-310.pyc


/home/xilinx/jupyter_notebooks/qick/qick_lib/qick/__init__.py


'/home/xilinx/qick_pkg.zip'

In [3]:
import sys, platform
print(sys.version)          # Python version
print(sys.executable)       # exact interpreter this kernel uses
print(sys.prefix)           # environment root (a venv path, not a conda path)
print(platform.machine())   # aarch64 on the ZCU216's ARM cores

3.8.2 (default, Mar 13 2020, 10:14:16) 
[GCC 9.3.0]
/usr/local/share/pynq-venv/bin/python3
/usr/local/share/pynq-venv
aarch64


In [4]:
from qick import QickSoc
import os
from pathlib import Path

base = Path("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_rfbv2_16mixmux16")
out_dir = base / "out"

# 1. Clean up the non-working links
for name in ["qick_216_rfbv2.bit", "qick_216_rfbv2.hwh"]:
    f = out_dir / name
    if f.exists() or f.is_symlink():
        f.unlink()

# 2. Create actual, native OS-level relative symlinks
os.symlink("../top/top.runs/impl_1/d_1_wrapper.bit", str(out_dir / "qick_216_rfbv2.bit"))
os.symlink("../top/top.gen/sources_1/bd/d_1/hw_handoff/d_1.hwh", str(out_dir / "qick_216_rfbv2.hwh"))

print("Native Linux symlinks successfully recreated!")

# Load the firmware bitstream
soc = QickSoc("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_rfbv2_16mixmux16/out/qick_216_rfbv2.bit", external_clk=True)

soccfg = soc
print(soccfg)

out = Path("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_rfbv2_16mixmux16/out")
for name in ["qick_216_rfbv2.bit", "qick_216_rfbv2.hwh"]:
    p = out / name
    if p.is_symlink():
        target = os.readlink(p)   # the link's path text, e.g. ../top/...
        p.unlink()                # removes only the link; the real file in top/ is untouched
        p.write_text(target)      # plain text placeholder, same as on the PC

Native Linux symlinks successfully recreated!
configuring reference clock chips, as requested
LMK04828 clock reference = 245.760 MHz, LMX2594 clock synth = 491.520 MHz
QICK running on ZCU216, software version 0.2.422

Firmware configuration (built Tue Oct 13 22:06:33 2026):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 0, DAC tile 1, DAC tile 2, DAC tile 3], [ADC tile 1, ADC tile 2]

	16 signal generator channels:
	0:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 1 is 1_228 on JHC2, or QICK box DAC port 1
	2:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 2 is 2_228 on JHC1, or QICK box DAC port 2
	3:	axis

In [5]:
# Jupyter setup boilerplate
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from qick import *
from qick.asm_v2 import AveragerProgramV2 # still necessary even after importing all from QICK

# useful libraries
import numpy as np
import random 
import time
import math

# MixMuxed Signal Generation - Long Period For Spectrum Analysis

Similiar to code from Sho's Github, and copied from MixMux8SG.ipynb but modified. Implementing / testing the multi-tone generators. Readout is declared for the sake of running but is not used.

In [6]:
class MixMuxProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        gen_ch = cfg['gen_ch']
        self.declare_gen(ch=gen_ch, nqz=1,
                         mixer_freq=cfg['mixer_freq'],
                         mux_freqs=cfg['pulse_freqs'],
                         mux_gains=cfg['pulse_gains'],
                         mux_phases=cfg['pulse_phases'])

        self.add_pulse(ch=gen_ch, name="mymixmux",
                       style="const",
                       length=cfg["pulse_len"],
                       mask=cfg['pulse_mask'])   # tones to play must be in the mask

    def _body(self, cfg):
        self.trigger(pins=[0], t=cfg['trig_time'])   # scope trigger only
        self.pulse(ch=cfg['gen_ch'], name="mymixmux", t=0)

In [39]:
config = {'gen_ch': 15,
          'mixer_freq': 0,
          'pulse_freqs': [199.2, 199.3, 199.4, 199.5, 199.6, 199.7, 199.8, 199.9,
                          200.0, 200.1, 200.2, 200.3, 200.4, 200.5, 200.6, 200.7],
          'pulse_gains': [0.0625]*16,
          'pulse_phases': [0]*16,
          'pulse_mask': list(range(16)),
          'trig_time': 0.4,
          'pulse_len': 4000000.0,   # us
         }

prog = MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
prog.run_rounds(soc, rounds=1, progress=True)

  0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
soc.reset_gens()